# Training Curves — NYUv2 Depth LoRA (50 epochs)
Parses `log.txt` (JSON Lines, one record per epoch) and plots loss + LR.
Re-run any cell to refresh while training is still running.

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

LOG_PATH = '../output/depth/nyu_rgb_r8/log.txt'

In [ ]:
# --- Parse log.txt (JSON Lines: one dict per epoch) ---
rows = []
with open(LOG_PATH) as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        try:
            rows.append(json.loads(line))
        except json.JSONDecodeError:
            pass  # skip malformed lines

df = pd.DataFrame(rows)
# rename for convenience
df = df.rename(columns={
    'train/lr':                   'lr',
    'train/loss':                 'train_loss',
    'train/loss_tok_depth@224':   'depth_loss',
    'train/grad_norm':            'grad_norm',
    'val/loss':                   'val_loss',
    'val/token_acc':              'val_acc',
})

print(f"Epochs in log : {len(df)}")
if not df.empty:
    last = df.iloc[-1]
    print(f"Last epoch    : {int(last.epoch)}")
    print(f"Train loss    : {last.train_loss:.4f}")
    print(f"LR            : {last.lr:.2e}")
    if 'val_loss' in last and pd.notna(last.get('val_loss')):
        print(f"Val loss      : {last.val_loss:.4f}")
df.head()

In [ ]:
# --- Plot: Loss curves ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('NYUv2 Depth — LoRA fine-tuning (rank=8, XL)', fontsize=13)

# Left: train loss + val loss by epoch
ax = axes[0]
ax.plot(df['epoch'], df['train_loss'], 'o-', color='steelblue', lw=2, ms=4, label='train loss')
val_df = df.dropna(subset=['val_loss'])
if not val_df.empty:
    ax.plot(val_df['epoch'], val_df['val_loss'], 's--', color='tomato', lw=2, ms=6, label='val loss')
ax.set_xlabel('Epoch')
ax.set_ylabel('Cross-entropy loss')
ax.set_title('Train vs Val loss')
ax.xaxis.set_major_locator(ticker.MaxNLocator(integer=True))
ax.legend()
ax.grid(True, alpha=0.3)

# Right: depth token loss only (the actual depth prediction loss)
ax2 = axes[1]
ax2.plot(df['epoch'], df['depth_loss'], 'o-', color='darkorchid', lw=2, ms=4, label='depth token loss')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Cross-entropy loss (depth tokens)')
ax2.set_title('Depth token loss')
ax2.xaxis.set_major_locator(ticker.MaxNLocator(integer=True))
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../output/depth/nyu_rgb_r8/loss_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → output/depth/nyu_rgb_r8/loss_curves.png')

In [ ]:
# --- Plot: Learning rate schedule ---
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(df['epoch'], df['lr'], 'o-', color='darkorange', lw=2, ms=4)
ax.set_xlabel('Epoch')
ax.set_ylabel('Learning rate')
ax.set_title('Learning rate schedule (cosine + warmup)')
ax.ticklabel_format(axis='y', style='sci', scilimits=(0, 0))
ax.xaxis.set_major_locator(ticker.MaxNLocator(integer=True))
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../output/depth/nyu_rgb_r8/lr_schedule.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → output/depth/nyu_rgb_r8/lr_schedule.png')